In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
sns.set_style("whitegrid")

print("All libraries loaded.")

All libraries loaded.


In [2]:
from src.data.loader import BookCrossingLoader
from src.data.preprocessor import BookCrossingPreprocessor

loader = BookCrossingLoader("../data/raw")
raw = loader.load_all(verbose=False)

prep = BookCrossingPreprocessor()
clean = prep.fit_transform(raw)

users   = clean.users
books   = clean.books
ratings = clean.ratings

print(f"Users:   {len(users):,}")
print(f"Books:   {len(books):,}")
print(f"Ratings: {len(ratings):,}")

Users:   278,859
Books:   271,378
Ratings: 1,149,753


In [3]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Build sparse matrix — rows = users, cols = books, values = ratings
sparse_matrix = prep.build_sparse_matrix(clean)
print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Non-zero entries:    {sparse_matrix.nnz:,}")
print(f"Sparsity:            {(1 - sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1])):.6%}")

Sparse matrix shape: (105283, 340054)
Non-zero entries:    1,149,753
Sparsity:            99.996789%


In [4]:
def find_similar_users(sparse_matrix, k=10, batch_size=500):
    """
    For every user, find the K most similar users using cosine similarity.
    Processes users in batches to avoid memory issues on large matrices.
    
    Parameters
    ----------
    sparse_matrix : csr_matrix
        User-book rating matrix (rows=users, cols=books).
    k : int
        Number of similar users to find per user. Default 10.
    batch_size : int
        Number of users to process per batch. Default 500.
    
    Returns
    -------
    dict
        Maps user_index -> {'similar_users': [...], 'similarity_scores': [...]}
    """
    n_users = sparse_matrix.shape[0]
    similar_users_dict = {}

    for start in tqdm(range(0, n_users, batch_size), desc="Finding similar users"):
        end = min(start + batch_size, n_users)
        batch = sparse_matrix[start:end]

        # Cosine similarity between this batch and all users
        sims = cosine_similarity(batch, sparse_matrix)

        for i, user_sims in enumerate(sims):
            user_idx = start + i

            # Exclude self
            user_sims[user_idx] = -np.inf

            # Get top K indices sorted by similarity descending
            top_k_idx = np.argsort(user_sims)[-k:][::-1]
            top_k_scores = user_sims[top_k_idx]

            # Only store if at least one similar user has nonzero similarity
            if top_k_scores.max() > 0:
                similar_users_dict[user_idx] = {
                    'similar_users':      top_k_idx.tolist(),
                    'similarity_scores':  top_k_scores.tolist()
                }

    return similar_users_dict

print("Function defined. Running similarity search — this will take several minutes...")
similar_users = find_similar_users(sparse_matrix, k=10, batch_size=500)
print(f"\nDone. Found similar users for {len(similar_users):,} users.")

Function defined. Running similarity search — this will take several minutes...


Finding similar users: 100%|██████████████████████████████████████████████████████████████████████| 211/211 [11:55<00:00,  3.39s/it]


Done. Found similar users for 64,053 users.


In [5]:
# Save so we don't have to recompute every time
with open('../data/processed/similar_users.pkl', 'wb') as f:
    pickle.dump(similar_users, f)

print("Saved to data/processed/similar_users.pkl")
print(f"Total users with similar users found: {len(similar_users):,}")

Saved to data/processed/similar_users.pkl
Total users with similar users found: 64,053


In [6]:
def get_recommendations(user_idx, similar_users_dict, sparse_matrix, 
                         idx_to_user, idx_to_isbn, k_recs=5):
    """
    Generate top-K book recommendations for a single user.
    
    Uses weighted average of similar users' ratings:
        score(book) = sum(sim * rating) / sum(sim)
    
    Parameters
    ----------
    user_idx : int
        Row index of the target user in the sparse matrix.
    similar_users_dict : dict
        Output of find_similar_users().
    sparse_matrix : csr_matrix
        Full user-book rating matrix.
    idx_to_user : dict
        Maps row index back to original user_id string.
    idx_to_isbn : dict
        Maps column index back to ISBN string.
    k_recs : int
        Number of recommendations to return. Default 5.
    
    Returns
    -------
    list of (isbn, weighted_score) tuples, sorted descending.
    """
    if user_idx not in similar_users_dict:
        return []

    sim_indices = similar_users_dict[user_idx]['similar_users']
    sim_scores  = similar_users_dict[user_idx]['similarity_scores']

    # Books already read by this user
    user_row  = sparse_matrix[user_idx]
    books_read = set(user_row.indices)

    # Accumulate weighted ratings across similar users
    numerator   = {}
    denominator = {}

    for sim_idx, sim_score in zip(sim_indices, sim_scores):
        if sim_score <= 0:
            continue
        sim_row = sparse_matrix[sim_idx]
        for book_idx, rating in zip(sim_row.indices, sim_row.data):
            if book_idx in books_read:
                continue
            if book_idx not in numerator:
                numerator[book_idx]   = 0.0
                denominator[book_idx] = 0.0
            numerator[book_idx]   += sim_score * rating
            denominator[book_idx] += sim_score

    # Compute weighted average scores
    scores = {
        book_idx: numerator[book_idx] / denominator[book_idx]
        for book_idx in numerator
        if denominator[book_idx] > 0
    }

    # Sort and return top K
    top_k = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k_recs]
    return [(idx_to_isbn.get(book_idx, str(book_idx)), score) for book_idx, score in top_k]


print("Recommendation function defined.")

Recommendation function defined.


In [7]:
idx_to_isbn = clean.idx_to_isbn
idx_to_user = clean.idx_to_user

# Build ISBN -> title lookup
isbn_to_title = books.set_index('isbn')['title'].to_dict()

records = []
for user_idx in tqdm(similar_users, desc="Generating recommendations"):
    user_id = idx_to_user.get(user_idx, str(user_idx))
    recs = get_recommendations(
        user_idx, similar_users, sparse_matrix,
        idx_to_user, idx_to_isbn, k_recs=5
    )
    for isbn, score in recs:
        records.append({
            'User_ID':              user_id,
            'Book_ID':              isbn,
            'Book_Title':           isbn_to_title.get(isbn, 'Unknown'),
            'Recommendation_Score': round(score, 4)
        })

recommendations_df = pd.DataFrame(records)
recommendations_df.to_csv('../outputs/Book_recommendation.csv', index=False)

print(f"\nRecommendations generated for {recommendations_df['User_ID'].nunique():,} users")
print(f"Total recommendation rows: {len(recommendations_df):,}")
print()
print(recommendations_df.head(10).to_string(index=False))

Generating recommendations: 100%|████████████████████████████████████████████████████████████| 64053/64053 [06:09<00:00, 173.43it/s]



Recommendations generated for 61,020 users
Total recommendation rows: 275,057

User_ID    Book_ID                                                              Book_Title  Recommendation_Score
 276736 345386476X                                  Die letzte Zauberin. Valorians Kinder.                  10.0
 276736 3404139178                             Das Lacheln der Fortuna: Historischer Roman                   9.0
 276736 3453061187                                                        Die Jury. Roman.                   8.0
 276736 3453195841 Das sÃ?ÃÂ¼Ã?Ã?e VermÃ?ÃÂ¤chtnis. Ein Fall fÃ?ÃÂ¼r Hilary Tamar.                   8.0
 276736 3548255558                                            Das Blut des Teufels. Roman.                   8.0
 276744 0380820145                                                         Friendship Cake                   0.0
 276744 0446678457                              Cane River (Oprah's Book Club (Paperback))                   0.0
 276744 15732297